In [19]:
import pandas as pd
from neo4j import GraphDatabase, basic_auth
import os
import re
from dotenv import load_dotenv
from pathlib import Path
import json
from urllib.request import urlretrieve
from typing import Any

# LlamaIndex + OpenAI
from llama_index.core import SimpleDirectoryReader
from llama_index.core import PropertyGraphIndex
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core.indices.property_graph import SimpleLLMPathExtractor, SchemaLLMPathExtractor

from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore

import nest_asyncio 
nest_asyncio.apply()

In [9]:
#define some constants
DATA_DIR = "data"

#load the environment variables
dotenv_path = Path('../.env')
load_dotenv(dotenv_path=dotenv_path)  # This line brings all environment variables from .env into os.environ

# Get variables
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USER = os.getenv('NEO4J_USER')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

# Connect to database
driver = GraphDatabase.driver(NEO4J_URI, auth=basic_auth(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()


The United Nations publishes annual reports related to the Sustainable Development Goals.

These reports include:

- an overview report
- a progress report from the Secretary General
- extended reports for each Goal

## UN SDG Reports

In [7]:
# Retrieve reports

report_catalog = "8.UN_reports_2024.json"
# report_catalog = "8.UN_reports_2024-extended.json"

catalog_path = os.path.join(DATA_DIR, report_catalog)
un_reports_dir = os.path.join(DATA_DIR, "8.UN_reports")
with open(catalog_path) as catalog_file:
  un_sdg_catalog = json.load(catalog_file)

# print(un_sdg_catalog)

for report in un_sdg_catalog['reports']:
  print(f"Downloading {report['title']}")
  filename = os.path.basename(report['url'])
  report['filename'] = filename
  urlretrieve(report['url'], os.path.join(un_reports_dir, filename))

print(f"Downloaded {len(un_sdg_catalog['reports'])} reports")


Downloaded 2 reports


## GraphRAG with LlamaIndex

In [10]:
llm = OpenAI(temperature=0, model="gpt-4")


In [11]:
def get_meta(file_path):
    basename = os.path.basename(file_path)
    entry = next(entry for entry in un_sdg_catalog['reports'] if entry['filename'] == basename)
    return entry

# for file in Path(un_reports_dir).glob('*.pdf'):
#     print(get_meta(file))


In [12]:
from llama_index.core import SimpleDirectoryReader

reader = SimpleDirectoryReader(input_dir=un_reports_dir, required_exts=[".pdf"], file_metadata=get_meta)
documents = reader.load_data()

print(f"Document count: {len(documents)}")
print(documents[0].text)

TextNode(id_='00855fa4-3766-4658-9214-5cdb9afac529', embedding=None, metadata={'page_label': '', 'file_name': 'The-Sustainable-Development-Goals-Report-2024.pdf', 'title': 'The Sustainable Development Goals Report 2024', 'year': 2024, 'language': 'en', 'url': 'https://unstats.un.org/sdgs/report/2024/The-Sustainable-Development-Goals-Report-2024.pdf', 'filename': 'The-Sustainable-Development-Goals-Report-2024.pdf'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='3c60f737-2b68-49f8-bc61-f39cfcd4adcb', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'page_label': '', 'file_name': 'The-Sustainable-Development-Goals-Report-2024.pdf', 'title': 'The Sustainable Development Goals Report 2024', 'year': 2024, 'langu

### KG using SimpleLLMPathExtractor

In [20]:
neo4j_graph_store = Neo4jPropertyGraphStore(
    username=NEO4J_USER, password=NEO4J_PASSWORD, url=NEO4J_URI
)

simple_kg_extractor = SimpleLLMPathExtractor(
    llm=llm, max_paths_per_chunk=10
)

simple_index = PropertyGraphIndex.from_documents(
    documents,
    llm=llm,
    embed_kg_nodes=False,
    kg_extractors=[simple_kg_extractor],
    property_graph_store=neo4j_graph_store,
    show_progress=True,
)


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('config' used by 'apoc.meta.graphSample' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL apoc.meta.graphSample() YIELD nodes, relationships RETURN nodes, [rel in relationships | {name:apoc.any.property(rel, 'type'), count: apoc.any.property(rel, 'count')}] AS relationships"
Extracting paths from text: 100%|██████████| 119/119 [05:59<00:00,  3.02s/it]
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is now deprecated. Use CALL (e, row) { ... }} {posit

### Connect Unstructured Text to Sustainable Goals

In [25]:
def attach_chunks_to_mentioned_goals(tx):
    cypher = """UNWIND range(1,18) as goal_num
                CALL (goal_num) {
                    MATCH (g:Goal), (c:Chunk) 
                    WHERE g.code = toString(goal_num)
                        AND c.text CONTAINS "Goal " + goal_num + " "
                    MERGE (c)-[r:MENTIONS]->(g)
                    RETURN count(r) AS goal_mentions
                }
                RETURN goal_num, goal_mentions
                """
    result = tx.run(cypher)
    return result.data()

with driver.session() as session:
    totals = session.execute_write(attach_chunks_to_mentioned_goals)
    print('Total mentions...')
    print(totals)

Total mentions...
[{'goal_num': 1, 'goal_mentions': 2}, {'goal_num': 2, 'goal_mentions': 2}, {'goal_num': 3, 'goal_mentions': 4}, {'goal_num': 4, 'goal_mentions': 5}, {'goal_num': 5, 'goal_mentions': 2}, {'goal_num': 6, 'goal_mentions': 3}, {'goal_num': 7, 'goal_mentions': 4}, {'goal_num': 8, 'goal_mentions': 4}, {'goal_num': 9, 'goal_mentions': 2}, {'goal_num': 10, 'goal_mentions': 2}, {'goal_num': 11, 'goal_mentions': 2}, {'goal_num': 12, 'goal_mentions': 3}, {'goal_num': 13, 'goal_mentions': 2}, {'goal_num': 14, 'goal_mentions': 2}, {'goal_num': 15, 'goal_mentions': 3}, {'goal_num': 16, 'goal_mentions': 2}, {'goal_num': 17, 'goal_mentions': 2}, {'goal_num': 18, 'goal_mentions': 0}]
